In [ ]:
!pip install torch transformers

In [ ]:
from transformers import pipeline
import re
import datetime

llm = pipeline("text2text-generation",
    model="google/flan-t5-large",
    max_length=120)


print("Agentic AI Productivity Assistant")
print("Type 'exit' to quit\n")

def calculator_tool(text):
    """Calculate simple math expressions from user input."""
    try:
        expression = re.findall(r"[0-9]+|[\+\-\*/\.]", text)
        return str(eval("".join(expression)))
    except:
        return "I couldn't calculate that. Please check the expression."

def time_tool():
    """Return current system time."""
    return datetime.datetime.now().strftime("%H:%M:%S")

def productivity_tool(text):
    return (
        "- Divide the syllabus into 7 daily portions\n"
        "- Study difficult topics in the morning when focus is high\n"
        "- Practice questions immediately after studying each topic\n"
        "- Revise previous topics every evening\n"
        "- Take proper breaks and sleep well to retain information"
    )

def chat_tool(text):
    """Generates a general response using the LLM."""
    # Use the pre-initialized LLM to generate a response for general questions.
    # The max_length is already set during llm initialization, but can be
    # explicitly passed here for clarity or to override if needed.
    # Adding num_beams and early_stopping for better generation quality.
    response = llm(text, max_length=120, num_beams=4, early_stopping=True)[0]["generated_text"]
    return response

def decide_and_act(user_input):
    """
    AI agent decides which tool to use based on user intent.
    """
    prompt = f"""
You are an AI agent with access to tools:
1. calculator
2. time checker
3. productivity advisor

Rules:
- If the input contains math → TOOL_CALCULATOR
- If the input asks for current time → TOOL_TIME
- If the input is about study, exams, planning, deadlines, productivity → TOOL_PRODUCTIVITY
- Otherwise → ANSWER

Respond with only one:
TOOL_CALCULATOR
TOOL_TIME
TOOL_PRODUCTIVITY
ANSWER

User Input: {user_input}
Agent Decision:
"""
    decision = llm(prompt)[0]["generated_text"].strip()

    if decision == "TOOL_CALCULATOR":
        return calculator_tool(user_input)
    elif decision == "TOOL_TIME":
        return time_tool()
    elif decision == "TOOL_PRODUCTIVITY":
        return productivity_tool(user_input)
    else:
        return chat_tool(user_input)

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Agent: Goodbye!")
        break
    result = decide_and_act(user_input)
    print("Agent:", result)